# Run PeerConf — Kaggle 2× T4

Mofe's PeerConf code, unmodified, on Kaggle's free GPUs.

## Three things first, or nothing works

1. **Settings → Accelerator → GPU T4 × 2**
2. **Settings → Internet → On** (needs phone verification on your Kaggle account)
3. **Run → Factory reset** if you have already installed vLLM in this session

## Two things to know

Current vLLM (0.26) needs GPU compute capability 8.0 and the T4 is 7.5, which is the
"Engine core initialization failed" crash. Pinning 0.9.2 with `VLLM_USE_V1=0` selects the
older engine, which supports Turing.

Two T4s cannot feed 16 traces of 30,000 tokens, so the race is shrunk to fit. **These
numbers are a working-code check, not a result** — the real run needs a 48 GB card.

## 1. GPUs

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv
import torch, sys
print("GPUs:", torch.cuda.device_count(), "| python:", sys.executable)


## 2. Install vLLM

5–10 minutes. Complaints about `cuml`, `cudf`, `rmm`, `google-adk` are Kaggle's own
packages and are unrelated — ignore them.

`dynasor` is installed from GitHub, not PyPI: Mofe's script imports
`dynasor.core.evaluator.math_equal`, and the identically-named package on PyPI is a
different project that does not have it.

`transformers` is pinned too: vLLM 0.9.2 registers a config called `aimv2`, and newer
transformers already has one, which collides and kills the server at startup.

In [ ]:
import sys
!{sys.executable} -m pip install -q "vllm==0.9.2" "transformers==4.53.2" "git+https://github.com/hao-ai-lab/Dynasor.git" 2>&1 | grep -viE "cuml|cudf|rmm|pylibraft|cuvs|libcu|google-|grpcio|gradio|opentelemetry|protobuf" | tail -12


In [ ]:
import sys
!{sys.executable} -c "import vllm, torch, transformers; from dynasor.core.evaluator import math_equal; print('vllm', vllm.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, '| dynasor ok')"


## 3. Get the code

In [ ]:
import os, subprocess
REPO = "/kaggle/working/Algoverse-AI-Research"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/yityler/Algoverse-AI-Research.git", REPO],
                   check=True)
%cd /kaggle/working/Algoverse-AI-Research/peerconf
print(sorted(os.listdir(".")))


## 4. Start the model server

Downloads ~16 GB the first time, so expect 10–20 minutes. The cell prints progress from
the server log while it waits, and prints the cause immediately if it dies.

In [ ]:
import subprocess, time, requests, os, sys, threading

MODEL  = "deepseek-ai/DeepSeek-R1-0528-Qwen3-8B"
SERVER = "http://localhost:8000"
LOG    = "/kaggle/working/vllm_server.log"
os.makedirs("peerconf_out", exist_ok=True)

env = dict(os.environ, VLLM_USE_V1="0")   # older engine: the one that supports Turing

proc = subprocess.Popen(
    [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
     "--model", MODEL,
     "--port", "8000",
     "--tensor-parallel-size", "2",
     "--dtype", "half",
     "--max-model-len", "4096",
     "--gpu-memory-utilization", "0.90",
     "--max-logprobs", "20"],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT, env=env)

def tail():
    seen = 0
    while proc.poll() is None:
        try:
            lines = open(LOG).readlines()
            for l in lines[seen:]:
                if any(k in l.lower() for k in
                       ("loading", "error", "traceback", "capability", "started", "it/s")):
                    print("   ", l.rstrip()[:150], flush=True)
            seen = len(lines)
        except FileNotFoundError:
            pass
        time.sleep(15)

threading.Thread(target=tail, daemon=True).start()

t0 = time.time()
while True:
    try:
        if requests.get(f"{SERVER}/health", timeout=5).status_code == 200:
            print(f"\nSERVER UP after {time.time()-t0:.0f}s"); break
    except requests.exceptions.RequestException:
        pass
    if proc.poll() is not None:
        print("\nSERVER DIED. Most likely cause:")
        hits = subprocess.run(
            ["grep", "-iE", "compute capability|not supported|unsupported|no kernel"
             "|out of memory|ValueError|RuntimeError|AssertionError", LOG],
            capture_output=True, text=True).stdout.splitlines()
        print("\n".join(hits[:15]) or "(nothing obvious - run the log cell below)")
        raise RuntimeError("server died")
    if time.time() - t0 > 2700:
        raise RuntimeError("no server after 45 min")
    time.sleep(10)


In [ ]:
# the full server log, any time you want it
!tail -60 /kaggle/working/vllm_server.log


## 5. Size the race to fit

Writes a smaller copy of Mofe's script. **His file is not modified** — this only changes
the numbers at the top of a copy, and prints back what actually landed.

`WARMUP_MODE` is left alone at its default of `False`, so this is plain PeerConf.

In [ ]:
import re

SMALL = {
    "QIDS":               "range(2)",   # 2 questions instead of 30
    "SEATS":              "6",          # 6 concurrent instead of 16
    "MAX_TRACES":         "10",         # launch cap instead of 32
    "MAX_TOK_TRACE":      "3000",       # instead of 30000
    "WINDOW":             "512",        # must stay well under MAX_TOK_TRACE
    "DWELL_TOKENS":       "64",
    "FINAL_CHECK_TOKENS": "800",
}

src = open("cell2_race.py").read()
for k, v in SMALL.items():
    src = re.sub(rf"^{k}(\s*)=\s*\S+", f"{k}\\1= {v}", src, count=1, flags=re.M)
open("run_peerconf.py", "w").write(src)

for k in list(SMALL) + ["WARMUP_MODE", "LINE_TOP"]:
    m = re.search(rf"^{k}\s*=\s*(\S+)", src, re.M)
    print(f"  {k:20s} = {m.group(1) if m else '(not found!)'}")


## 6. Run it

A few minutes on 2× T4. Full output, not truncated.

In [ ]:
import sys
!{sys.executable} run_peerconf.py


## 7. Results

In [ ]:
import pickle, glob

files = sorted(glob.glob("peerconf_out/*.pkl"))
print(f"{len(files)} question(s)\n")
tot_tok = tot_ok = 0
for p in files:
    d = pickle.load(open(p, "rb"))
    ans = d["voting"]["majority"][0]
    ok  = str(ans).strip() == str(d["gt"]).strip()
    tot_tok += d["tokens"]; tot_ok += ok
    print(f"  Q{d['qid']:<3} answer={str(ans):<10} truth={str(d['gt']):<10} "
          f"{'CORRECT' if ok else 'wrong':<8} tokens={d['tokens']:,} "
          f"launched={d['launched']}")
if files:
    print(f"\n  total tokens {tot_tok:,} | {tot_tok//len(files):,} per question "
          f"| {tot_ok}/{len(files)} correct")
    print("\n  2 questions on shrunken settings is a working-code check, not a result.")


## 8. Optional — the confidence charts

Mofe's `cell3` plots each race. Saves PNGs into `peerconf_out/`.

In [ ]:
import sys
!{sys.executable} cell3_confidence_timeline.py
